In [1]:
%pip install faster-whisper ollama

  Using cached faster_whisper-1.2.1-py3-none-any.whl.metadata (16 kB)
  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached huggingface_hub-1.5.0-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached hf_xet-1.3.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
Using cached faster_whisper-1.2.1-py3-none-any.whl (1.1 MB)
Using cached ollama-0.6.1-py3-none-any.whl (14 kB)
   ---------------------------------------- 0.0/31.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.8 MB


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from faster_whisper import WhisperModel
import ollama
import json
import sqlite3
import time

In [16]:
pip install torch

^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/113.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/113.7 MB 2.1 MB/s eta 0:00:54
   ---------------------------------------- 1.0/113.7 MB 1.9 MB/s eta 0:00:59
   ---------------------------------------- 1.3/113.7 MB 1.9 MB/s eta 0:00:59
    --------------------------------------- 1.6/113.7 MB 1.7 MB/s eta 0:01:06
    --------------------------------------- 1.8/113.7 MB 1.7 MB/s eta 0:01:06
    --------------------------------------- 2.1/113.7 MB 1.6 MB/s eta 0:01:09
    --------------------------------------- 2.4/113.7 MB 1.4 MB/s eta 0:01:20
    --------------------------------------- 2.4/113.7 MB 1.4 MB/s eta 0:01:20
    --------------------------------------- 2.6/113.7 MB 1.2 MB/s eta 0:01:30
    --------------------------------------- 2.6/113.7 MB 1.2 MB/s eta 0:01:30
   - -------------------------------------- 2.9/113.7 MB 1.1 MB/s eta 0:01:40


In [12]:
from faster_whisper import WhisperModel
import ctypes

def gpu_usable():
    try:
        ctypes.CDLL("nvcuda.dll")      # NVIDIA driver
        ctypes.CDLL("cublas64_12.dll") # CUDA math library
        return True
    except:
        return False

if gpu_usable():
    device = "cuda"
    compute_type = "float16"
else:
    device = "cpu"
    compute_type = "int8"

print(f"Using device: {device} | compute_type: {compute_type}")

model = WhisperModel(
    "base",
    device=device,
    compute_type=compute_type,
    num_workers=1
)

print("✅ Whisper loaded successfully!")

Using device: cpu | compute_type: int8
✅ Whisper loaded successfully!


In [24]:
from datetime import datetime, timedelta

today = datetime.now().date()
TODAY_STR = today.strftime("%d %B %Y")
TODAY_ISO = today.strftime("%Y-%m-%d")   # AI understands this MUCH better
print("Today:", TODAY_STR)

Today: 27 February 2026


In [17]:
def transcribe_audio(file_path):
    """Fast transcription with optimized settings."""
    start = time.time()
    
    segments, info = model.transcribe(
        file_path,
        beam_size=1,          # ✅ FIX 4: beam_size=1 (greedy) is ~3x faster than default 5
        language="en",        # ✅ FIX 5: Set language to skip auto-detection overhead
        vad_filter=True,      # ✅ FIX 6: Skip silent parts automatically
        vad_parameters=dict(
            min_silence_duration_ms=500  # skip pauses >0.5s
        ),
        condition_on_previous_text=False,  # ✅ FIX 7: Faster, avoids compounding errors
        temperature=0.0,      # greedy decoding, no sampling overhead
    )
    
    transcript_parts = []
    for segment in segments:
        transcript_parts.append(segment.text)
    
    transcript = " ".join(transcript_parts).strip()
    
    elapsed = time.time() - start
    print(f"✅ Transcription done in {elapsed:.1f}s | Audio length: {info.duration:.1f}s")
    print(f"   Speed ratio: {info.duration/elapsed:.1f}x realtime")
    
    return transcript

In [25]:
import ollama
from datetime import datetime, timedelta

def generate_summary(transcript):

    max_chars = 12000
    if len(transcript) > max_chars:
        transcript = transcript[:max_chars] + "... [truncated]"

    prompt = f"""
You are a professional Project Manager and Scheduler.

TODAY'S DATE (ISO): {TODAY_ISO}
TODAY (READABLE): {TODAY_STR}

Your job is to convert the meeting discussion into a REAL project execution schedule.

CRITICAL REQUIREMENTS:
• Use REAL calendar dates only (YYYY-MM-DD)
• NEVER use: 'next week', '2 weeks', 'few days', 'soon'
• Every task MUST have:
  - start_date
  - end_date
  - duration_days
• Tasks must be ordered logically (dependencies)
• High priority tasks should happen earlier
• All tasks must complete BEFORE the next meeting date
• The next meeting must be scheduled after major tasks finish
• Assume a standard 5-day working week (Mon–Fri)
• Do not schedule work on weekends

You must create a structured plan like a mini project timeline.

Return ONLY valid JSON:

{{
  "project_summary": "",
  "next_meeting_date": "YYYY-MM-DD",
  "tasks_schedule": [
    {{
      "task": "",
      "owner_role": "",
      "priority": "High/Medium/Low",
      "start_date": "YYYY-MM-DD",
      "end_date": "YYYY-MM-DD",
      "duration_days": number,
      "dependency": "task name or null"
    }}
  ],
  "meeting_agenda": [
    "",
    "",
    "",
    ""
  ],
  "estimated_total_project_completion_date": "YYYY-MM-DD",
  "followup_meetings_required": number
}}

Transcript:
{transcript}
"""

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": 0.15,
            "num_predict": 1000,
            "top_k": 25
        }
    )

    return response['message']['content']

In [22]:
import json
import re

def parse_json(output):
    try:
        json_text = re.search(r'\{.*\}', output, re.DOTALL).group()
        return json.loads(json_text)
    except:
        print("⚠️ Failed to parse JSON. Raw output:\n", output)
        return None

In [6]:
def init_db():
    conn = sqlite3.connect("meetings.db")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS meetings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            transcript TEXT,
            summary TEXT
        )
    """)
    conn.commit()
    conn.close()

def save_meeting(transcript, summary):
    conn = sqlite3.connect("meetings.db")
    conn.execute(
        "INSERT INTO meetings (transcript, summary) VALUES (?, ?)",
        (transcript, json.dumps(summary))
    )
    conn.commit()
    conn.close()

In [26]:
# ▶️ CHANGE THIS PATH TO YOUR AUDIO FILE
audio_file = r"C:\\Users\\mayan\Downloads\\1m.mpeg"

total_start = time.time()

print("🎙️ Step 1: Transcribing audio...")
transcript = transcribe_audio(audio_file)

print("🤖 Step 2: Generating summary...")
llm_output = generate_summary(transcript)
structured = parse_json(llm_output)

if structured:
    init_db()
    save_meeting(transcript, structured)
    print(f"\n✅ Done! Total time: {time.time()-total_start:.1f}s")
    
    # 📋 FULL TRANSCRIPT
    print("\n" + "="*60)
    print("📝 FULL TRANSCRIPT")
    print("="*60)
    print(transcript)
    
    # 📊 STRUCTURED SUMMARY
    print("\n" + "="*60)
    print("📊 STRUCTURED SUMMARY")
    print("="*60)
    print(json.dumps(structured, indent=2))

else:
    print("⚠️ Could not parse JSON. Raw LLM output:")
    print(llm_output)
    print("\n📝 FULL TRANSCRIPT:")
    print(transcript)

🎙️ Step 1: Transcribing audio...
✅ Transcription done in 6.8s | Audio length: 98.4s
   Speed ratio: 14.5x realtime
🤖 Step 2: Generating summary...


KeyboardInterrupt: 

In [15]:
segments, info = model.transcribe(r"C:\\Users\\mayan\\Downloads\\1m.mpeg")

transcript = ""
for segment in segments:
    transcript += segment.text + " "

print(transcript)

 Hello, everyone. Thank you guys for coming to our weekly student success meeting and let's just get started  So I have our list of chronically absent students here, and I've been noticing a troubling trend  A lot of students are skipping on Fridays. Does anyone have any idea what's going on?  I've heard some of my mentees talking about how it's really hard to get out of bed on Fridays  It might be good if we did something like a pancake breakfast to encourage them to come  I think that's a great idea. Let's try that next week  It might also be because a lot of students have been getting sick now that it's getting colder outside  I've had a number of students come by my office with symptoms like sniffling and coughing  We should put up posters with tips for not getting sick since it's almost flu season  Like you know wash your hands after the bathroom stuff like that  I think that's a good idea and it'll be a good reminder for the teachers as well  So one other thing I wanted to talk a

In [27]:
import ollama
from datetime import datetime, timedelta

def generate_summary(transcript):

    max_chars = 12000
    if len(transcript) > max_chars:
        transcript = transcript[:max_chars] + "... [truncated]"

    prompt = f"""
You are a professional Project Manager and Scheduler.

TODAY'S DATE (ISO): {TODAY_ISO}
TODAY (READABLE): {TODAY_STR}

Your job is to convert the meeting discussion into a REAL project execution schedule.

CRITICAL REQUIREMENTS:
• Use REAL calendar dates only (YYYY-MM-DD)
• NEVER use: 'next week', '2 weeks', 'few days', 'soon'
• Every task MUST have:
  - start_date
  - end_date
  - duration_days
• Tasks must be ordered logically (dependencies)
• High priority tasks should happen earlier
• All tasks must complete BEFORE the next meeting date
• The next meeting must be scheduled after major tasks finish
• Assume a standard 5-day working week (Mon–Fri)
• Do not schedule work on weekends

You must create a structured plan like a mini project timeline.

Return ONLY valid JSON:

{{
  "project_summary": "",
  "next_meeting_date": "YYYY-MM-DD",
  "tasks_schedule": [
    {{
      "task": "",
      "owner_role": "",
      "priority": "High/Medium/Low",
      "start_date": "YYYY-MM-DD",
      "end_date": "YYYY-MM-DD",
      "duration_days": number,
      "dependency": "task name or null"
    }}
  ],
  "meeting_agenda": [
    "",
    "",
    "",
    ""
  ],
  "estimated_total_project_completion_date": "YYYY-MM-DD",
  "followup_meetings_required": number
}}

Transcript:
{transcript}
"""

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": 0.15,
            "num_predict": 1000,
            "top_k": 25
        }
    )

    return response['message']['content']

In [28]:
report = meeting_planner(transcript)

print(report)

with open("meeting_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

 ==== MEETING SUMMARY =====
Topic: Discussion on strategies to reduce chronic student absences
Short Summary: The meeting addressed the issue of students frequently missing school, particularly on Fridays, and proposed solutions such as organizing a pancake breakfast, creating health awareness posters, and offering support for students dealing with personal issues.

===== ACTION ITEMS =====
• Organize a pancake breakfast - Meeting Project Manager - High Priority - Next Week
• Create health awareness posters - Graphics Designer - Medium Priority - Two weeks
• Assist John Smith (mentioned student) with family's childcare needs - Meeting Project Manager - High Priority - Immediate Action

===== NEXT MEETINGS PLAN =====
Number of meetings required: 2
Reason: Follow-up on action items and discuss any new concerns or strategies.

===== SCHEDULE SUGGESTION =====
Meeting 1: Current Date (Discussion and Action Planning)
Meeting 2: Two weeks from now (Follow-up and Progress Update)

===== NEXT M